# Experimentos TDAH — una corrida

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpospinalo/tdha-revision/blob/main/tdha_experimentos.ipynb)

Este notebook ejecuta **una sola corrida**: un sitio, un subconjunto de ROIs, un
enventanado y una arquitectura. Produce un CSV con las 50 repeticiones de la
validación cruzada y sube los resultados al repositorio.

## Cómo trabaja el equipo

Cada persona corre configuraciones distintas y hace push de sus resultados. No hay que
coordinarse ni esperar a nadie: cada corrida escribe en su propia carpeta, cuyo nombre
incluye un identificador derivado de la configuración, así que **los push nunca chocan
entre sí**. Al final se compilan todas las corridas juntas.

Para tomar una tarea: editar la celda de configuración, ejecutar el notebook completo
y hacer push.

## Reglas

1. **No cambiar `SEED`.** El valor 42 es lo que garantiza que todas las corridas usen
   exactamente las mismas particiones y que la comparación entre ellas sea pareada.
2. **Editar solo la celda de configuración.** Toda la lógica está en `src/`,
   versionada en el repositorio.
3. **No editar a mano ningún CSV de `results/`.** Si un resultado está mal, se vuelve
   a correr.
4. Si la corrida falla a la mitad no hay nada que limpiar: no se sube nada hasta la
   última celda.

## 1. Preparar el entorno

In [ ]:
# Verificar que hay GPU asignada.
# Si no aparece nada: Entorno de ejecución > Cambiar tipo de entorno > GPU
!nvidia-smi -L || echo "SIN GPU — la corrida será mucho más lenta"

In [ ]:
import os, subprocess

REPO = "https://github.com/jpospinalo/tdha-revision.git"
ROOT = "/content/" + REPO.rstrip("/").split("/")[-1].replace(".git", "")

if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)

subprocess.run(["pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
os.chdir(f"{ROOT}/src")
print("listo:", os.getcwd())

El repositorio pesa unos 40 MB y no usa Git LFS: solo versiona las señales BOLD. Los
tensores de conectividad se derivan al vuelo en cada corrida.

## 2. Configuración

**Esta es la única celda que hay que editar.**

| Parámetro | Valores |
|---|---|
| `SITIO` | `NYU`, `Peking`, `NeuroIMAGE`, `OHSU` |
| `ROI_SET` | `12`, `18`, `39`, `116` |
| `MODELO` | `lstm`, `gru`, `cnn1d`, `transformer` |
| `VENTANA` / `PASO` | enventanado deslizante, en TR |

Advertencias por sitio, que el script también verifica:

- **OHSU** tiene 74 TR por sujeto: con ventana 70 solo caben 3 ventanas. Bajar
  `N_SPLITS` a 5 y considerar una ventana más corta.
- **NeuroIMAGE** tiene 39 sujetos: bajar `N_SPLITS` a 5.
- **Peking** está desbalanceado (109 / 74): activar `CLASS_WEIGHT`.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN DE ESTA CORRIDA
# ---------------------------------------------------------------------------
SITIO        = "NYU"
ROI_SET      = "12"
MODELO       = "lstm"
MODEL_ARG    = []        # p. ej. ["units=64", "dropout=0.2"]

VENTANA      = 70        # longitud de la ventana, en TR
PASO         = 2         # desplazamiento entre ventanas, en TR

N_SPLITS     = 10        # bajar a 5 en NeuroIMAGE y OHSU
N_REPEATS    = 5         # 10 x 5 = 50 repeticiones
CLASS_WEIGHT = False     # activar en Peking

SEED         = 42        # NO CAMBIAR
# ---------------------------------------------------------------------------

print(f"{SITIO} · {ROI_SET} ROIs · ventana {VENTANA}/{PASO} · {MODELO} "
      f"{MODEL_ARG or ''} · {N_SPLITS}x{N_REPEATS} = {N_SPLITS * N_REPEATS} repeticiones")

In [ ]:
# Subconjuntos de ROIs y arquitecturas disponibles.
!python run_experiment.py --list-roi-sets
!python run_experiment.py --list-models

## 3. Comprobaciones previas

`--dry-run` valida los datos y las particiones sin entrenar. Conviene mirar el número
de ventanas, que no aparezcan avisos, y anotar la `huella`: dos corridas solo son
comparables si coinciden su sitio, su semilla y esa huella.

In [ ]:
import subprocess


def ejecutar(extra=(), capturar=False):
    """Invoca run_experiment.py con la configuración de este notebook."""
    cmd = ["python", "run_experiment.py",
           "--site", SITIO, "--roi-set", str(ROI_SET), "--model", MODELO,
           "--window", str(VENTANA), "--step", str(PASO), "--seed", str(SEED),
           "--n-splits", str(N_SPLITS), "--n-repeats", str(N_REPEATS)]
    if MODEL_ARG:
        cmd += ["--model-arg"] + list(MODEL_ARG)
    if CLASS_WEIGHT:
        cmd += ["--class-weight"]
    cmd += list(extra)

    if capturar:
        r = subprocess.run(cmd, capture_output=True, text=True)
        return r.returncode, r.stdout + r.stderr

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    salida = []
    for linea in proc.stdout:
        print(linea, end="", flush=True)
        salida.append(linea)
    return proc.wait(), "".join(salida)


_, texto = ejecutar(["--dry-run"], capturar=True)
print(texto)

## 4. Prueba de humo

Dos pliegues y tres épocas, para verificar que todo el camino funciona antes de lanzar
una corrida de varias horas. Escribe en `/tmp`, así que no ensucia los resultados ni
se sube nada.

In [ ]:
rc, _ = ejecutar(["--n-splits", "2", "--n-repeats", "1",
                  "--epochs", "3", "--patience", "2",
                  "--out", "/tmp/smoke", "--tag", "smoke",
                  "--overwrite", "--verbose"])
print("\nprueba de humo:", "correcta" if rc == 0 else f"FALLÓ (código {rc})")

## 5. La corrida

Aquí ocurre el trabajo. El tiempo depende sobre todo del subconjunto de ROIs: con 12
la LSTM tiene unos 100 mil parámetros; con 116, unos 3,5 millones sobre la misma
muestra.

Si Colab desconecta a mitad, se vuelve a ejecutar el notebook desde el principio.

In [ ]:
import re

rc, salida = ejecutar(["--verbose"])

# El identificador se imprime siempre, incluso si la corrida ya existía.
m = re.search(r"corrida:\s*(\S+)", salida)
RUN_ID = m.group(1) if m else None

if rc != 0 and "ESTA_CONFIGURACION_YA_SE_EJECUTO" in salida:
    print("\nEsta configuración ya estaba corrida. Se reutilizan sus resultados.")
elif rc != 0:
    raise RuntimeError(f"la corrida falló con código {rc}")
if not RUN_ID:
    raise RuntimeError("no se pudo determinar el identificador de la corrida")

print("\n" + "=" * 70)
print("identificador de la corrida:", RUN_ID)

## 6. Resultados

`metrics_val.csv` es el archivo principal: una fila por repetición de la validación
cruzada. Los demás permiten reconstruir después las curvas de convergencia, las
matrices de confusión y las curvas ROC sin reentrenar.

In [ ]:
import pandas as pd

RUTA = f"../results/runs/{RUN_ID}"
val = pd.read_csv(f"{RUTA}/metrics_val.csv")
tra = pd.read_csv(f"{RUTA}/metrics_train.csv")

print(f"{len(val)} repeticiones\n")
M = ["accuracy", "precision", "recall", "auc"]
resumen = pd.DataFrame({
    "entrenamiento": [tra[m].mean() * 100 for m in M],
    "validación":    [val[m].mean() * 100 for m in M],
    "desv. val.":    [val[m].std() * 100 for m in M],
}, index=M).round(2)
resumen["brecha"] = (resumen.entrenamiento - resumen["validación"]).round(2)
print(resumen.to_string())

In [ ]:
import glob, os

print("archivos generados:\n")
for f in sorted(glob.glob(f"{RUTA}/*")):
    extra = f"{len(pd.read_csv(f)):>6d} filas" if f.endswith(".csv") else ""
    print(f"  {os.path.basename(f):26s} {extra}")

In [ ]:
# La dispersión importa tanto como la media: con muestras pequeñas, repeticiones
# individuales muy altas reflejan particiones favorables, no mejor aprendizaje.
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.hist(val.accuracy * 100, bins=15, edgecolor="white")
a.axvline(val.accuracy.mean() * 100, color="crimson", label="media")
a.set_xlabel("accuracy de validación (%)"); a.set_ylabel("repeticiones"); a.legend()

hist = pd.read_csv(f"{RUTA}/history.csv")
for _, g in hist.groupby("fold"):
    b.plot(g.epoch, g.inner_val_loss, alpha=0.25, linewidth=0.8)
b.set_xlabel("época"); b.set_ylabel("pérdida (partición interna)")
b.set_title("convergencia por repetición")
plt.tight_layout(); plt.show()

## 7. Subir los resultados

El push necesita un token de acceso personal de GitHub. **No escribirlo en el
notebook**: se guarda en el panel de secretos de Colab (icono de la llave, a la
izquierda) con el nombre `GITHUB_TOKEN`, y desde ahí se lee sin que quede en el
archivo ni en el historial.

Para crearlo: GitHub → Settings → Developer settings → Personal access tokens →
Fine-grained tokens, con permiso de escritura sobre este repositorio.

In [ ]:
import subprocess, getpass

NOMBRE = "tu nombre"
CORREO = "tu.correo@ejemplo.com"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = getpass.getpass("token de GitHub: ")

for k, v in [("user.name", NOMBRE), ("user.email", CORREO)]:
    subprocess.run(["git", "config", k, v], cwd=ROOT, check=True)
print("identidad configurada")

In [ ]:
def git(*args):
    r = subprocess.run(["git"] + list(args), cwd=ROOT, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode


git("add", f"results/runs/{RUN_ID}")
git("commit", "-m", f"exp: {RUN_ID}")

# Traer primero lo que hayan subido los demás. Como cada corrida vive en su propia
# carpeta, no puede haber conflictos de fusión.
git("pull", "--no-rebase", "origin", "main")

# La URL con el token se arma aquí y no se imprime, para que no quede en la salida
# del notebook ni en la configuración del repositorio.
url = REPO.replace("https://", f"https://{TOKEN}@")
r = subprocess.run(["git", "push", url, "main"], cwd=ROOT,
                   capture_output=True, text=True)
print("push correcto" if r.returncode == 0
      else "FALLÓ el push:\n" + r.stderr.replace(TOKEN, "***"))

## 8. Compilar (cuando ya haya varias corridas)

No hace falta ejecutarlo en cada corrida. Reúne todo lo que haya en `results/runs/`,
lo resume en una tabla y **verifica que las corridas sean comparables**: avisa si
detecta semillas distintas, enventanados distintos, señales BOLD que cambiaron entre
corridas o resultados producidos con código sin confirmar.

In [ ]:
!python compile_results.py